In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df_clean = pd.read_csv(
    "../data/raw/PS_20174392719_1491204439457_log.csv"
)

print("Dataset Shape:", df_clean.shape)

Dataset Shape: (6362620, 11)


In [2]:
print("Missing Values:", df_clean.isnull().sum().sum())
print("Duplicate Rows:", df_clean.duplicated().sum())

print("Negative Amount:",
      (df_clean["amount"] < 0).sum())

print("Invalid Step:",
      (df_clean["step"] < 1).sum())

print("Invalid Fraud Label:",
      (~df_clean["isFraud"].isin([0, 1])).sum())

print("Invalid Transaction Type:",
      (~df_clean["type"].isin([
          "PAYMENT", "TRANSFER", "CASH_OUT",
          "DEBIT", "CASH_IN"
      ])).sum())

Missing Values: 0
Duplicate Rows: 0
Negative Amount: 0
Invalid Step: 0
Invalid Fraud Label: 0
Invalid Transaction Type: 0


In [3]:
df_clean["type"] = df_clean["type"].astype("category")

df_clean["step"] = df_clean["step"].astype("int16")

df_clean["isFraud"] = df_clean["isFraud"].astype("int8")

df_clean["isFlaggedFraud"] = (
    df_clean["isFlaggedFraud"].astype("int8")
)

print(df_clean.dtypes)

step                 int16
type              category
amount             float64
nameOrig            object
oldbalanceOrg      float64
newbalanceOrig     float64
nameDest            object
oldbalanceDest     float64
newbalanceDest     float64
isFraud               int8
isFlaggedFraud        int8
dtype: object


In [4]:
# Extract hour from transaction step
df_clean["hour_of_day"] = (
    (df_clean["step"] - 1) % 24
).astype("int8")

# Extract simulation day
df_clean["simulation_day"] = (
    (df_clean["step"] - 1) // 24 + 1
).astype("int16")

# Identify TRANSFER and CASH_OUT transactions
df_clean["is_transfer_or_cashout"] = (
    df_clean["type"]
    .isin(["TRANSFER", "CASH_OUT"])
    .astype("int8")
)

# Log transformation
df_clean["amount_log"] = np.log1p(
    df_clean["amount"]
)

display(df_clean.head())

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,hour_of_day,simulation_day,is_transfer_or_cashout,amount_log
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0,1,0,9.194276
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0,1,0,7.531166
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0,1,1,5.204007
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,0,1,1,5.204007
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0,1,0,9.364703


In [5]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_dir / "transactions_clean.csv",
    index=False
)

print("Preprocessing dataset completed!")

Preprocessing dataset completed!


In [6]:
print("Final Dataset Shape:", df_clean.shape)

print("\nNew Features Validation:")
print(
    df_clean[
        [
            "hour_of_day",
            "simulation_day",
            "is_transfer_or_cashout",
            "amount_log"
        ]
    ].isnull().sum()
)

print(
    "Infinite Values:",
    np.isinf(df_clean["amount_log"]).sum()
)

Final Dataset Shape: (6362620, 15)

New Features Validation:
hour_of_day               0
simulation_day            0
is_transfer_or_cashout    0
amount_log                0
dtype: int64
Infinite Values: 0
